# Where does the data come from?

## Reproducible data management for big neuroimaging

**AS4SAN Workshop**

---

## Every other module assumes the data is already here

> *"Download the dataset and place it in `./data/`"*
>
> — Every tutorial ever

- Preprocessing pipelines assume the files exist
- Analysis scripts assume a specific directory layout
- Papers reference "the dataset" but not *which version*

**Nobody talks about how the data got there — or whether it's the same data you had last week.**

## Does this look familiar?

```
my_study/
├── data_final/
├── data_final_v2/
├── data_FINAL_really/
├── analysis_old.R
├── analysis_new.R
├── analysis_new_fixed.R
├── results_fig1_BACKUP.png
└── README_ignore_this.txt
```

- Which version of the data produced which figure?
- What changed between "final" and "final_v2"?
- Could a colleague reproduce your analysis from this folder?

**Version control solves this — for code *and* data.**

## A quick intro to git

**git** is a tool that tracks changes to files over time — like "track changes" in Word, but for entire folders.

**What git gives you:**
- A complete history of every change
- The ability to go back to any previous version
- A record of *who* changed *what* and *when*
- Safe collaboration — no overwriting each other's work

**The basic cycle:**
```bash
# 1. Make changes to your files

# 2. Stage — choose what to save
git add my_analysis.R

# 3. Commit — save a snapshot
git commit -m "Fix outlier filter"

# 4. Push — share with collaborators
git push
```

## git is great — but it wasn't built for big data

| git handles well | git struggles with |
|---|---|
| Analysis scripts (R, Python) | MRI scans (hundreds of MB each) |
| Participant lists, metadata (CSV, TSV) | Large datasets (GBs to TBs) |
| Documentation and notes | Binary files that don't "diff" well |
| Configuration files | Data stored on remote servers |

**We need something that works *like* git, but can handle big neuroimaging data.**

That's where DataLad comes in.

## The concrete problem: MR-RATE

| | |
|---|---|
| **Patients** | 83,425 |
| **MRI volumes** | 705,254 |
| **Total size** | 8.1 TB |
| **Modalities** | T1, T2, FLAIR, SWI, MRA |
| **Extras** | radiology reports, brain masks, segmentations |

**You can't just download 8 TB.**

- Too large for a single machine
- Evolving — new releases, corrections
- Multi-site (Zurich, Istanbul, NVIDIA)
- You only need a *subset* for your analysis

*Hugging Face: `Forithmus/MR-RATE` — CC BY-NC-SA 4.0*

## DataLad: version control that works for data

```
┌─────────────────────────────────────────────┐
│              DataLad                        │
│  ┌──────────────┐    ┌──────────────────┐   │
│  │     git      │    │    git-annex     │   │
│  │  tracks:     │    │  tracks:         │   │
│  │  • metadata  │    │  • large files   │   │
│  │  • scripts   │    │  • where to get  │   │
│  │  • history   │    │    them from     │   │
│  └──────────────┘    └──────────────────┘   │
└─────────────────────────────────────────────┘
```

- **git** handles the small stuff (scripts, metadata, history)
- **git-annex** handles the big stuff (scans, volumes) — stores *pointers*, not the actual files
- **DataLad** wraps both so you don't need to think about the layers underneath

## The core loop: install → get → run → rerun

| Step | Command | What happens |
|------|---------|-------------|
| **install** | `datalad install -s <url>` | Clone metadata — no large files yet |
| **get** | `datalad get sub-01/anat/*.nii.gz` | Fetch only the files you need |
| **run** | `datalad run -m "msg" -- <cmd>` | Execute & record what you did |
| **rerun** | `datalad rerun` | Reproduce from the recorded history |

**Let's try it live!**

---

# Live demo: the DataLad core loop

We'll walk through the four steps using a real dataset from OpenNeuro.

### Step 0: Install DataLad and git-annex

In [ ]:
# Install DataLad and git-annex (Colab doesn't have these by default)
!pip install -q datalad
!apt-get -qq install git-annex > /dev/null 2>&1

# Verify installation
!datalad --version
!git-annex version --raw

### Step 1: `datalad install` — clone metadata only

This clones the full directory tree and metadata, but **no large files**. It's fast even for terabyte-scale datasets.

In [ ]:
!datalad install -s https://github.com/OpenNeuroDatasets/ds000101.git ds000101-demo

In [ ]:
# Look at the directory structure — all the files are listed,
# but the large ones haven't been downloaded yet
import os
os.chdir('ds000101-demo')
!ls -la sub-01/anat/

Notice the file is a symlink — the actual data isn't here yet. DataLad knows *about* the file (its size, checksum, where to get it) but hasn't downloaded the content.

### Step 2: `datalad get` — fetch only the files you need

Instead of downloading the entire dataset, we fetch just one file.

In [ ]:
!datalad get sub-01/anat/sub-01_T1w.nii.gz

In [ ]:
# Now the file is real — we can check its size
!ls -lh sub-01/anat/sub-01_T1w.nii.gz

### Step 3: `datalad run` — execute and record provenance

This wraps a command so that DataLad records exactly what was run, on which inputs, producing which outputs — like a lab notebook for your computer.

In [ ]:
!datalad run -m "Compute file size summary" \
  --output summary.txt \
  -- bash -c 'wc -c sub-01/anat/sub-01_T1w.nii.gz > summary.txt'

print("\n--- Result ---")
!cat summary.txt

In [ ]:
# The provenance is now recorded in git history
!git log --oneline -3

### Step 4: `datalad rerun` — reproduce from recorded provenance

Replay the exact same command from the history — same inputs, same command, verifiable output.

In [ ]:
!datalad rerun

In [ ]:
# Same result — reproducible!
!cat summary.txt

---

## Why it matters

### Provenance — your digital lab notebook

> *"Which version of which data produced this figure?"*

DataLad records the **exact command**, **inputs**, and **outputs** — automatically.

### Selective access — download only what you need

Get 50 MB of scans for your analysis, not 8 TB for the whole dataset.

### Collaboration — no more emailing files

Everyone works from the same dataset. The history shows who changed what and when.

---

## On-ramp: getting the workshop dataset

Now let's get the dataset you'll use for the rest of the workshop.

In [ ]:
# Go back to the root directory
os.chdir('/content')

# Clone the workshop dataset (metadata only — fast)
!datalad install -s https://github.com/forithmus/MR-RATE workshop-data

In [ ]:
# Explore the dataset structure
os.chdir('workshop-data')
!ls

In [ ]:
# Fetch specific files you need for the exercises
# (uncomment and modify the path for the files you actually need)
# !datalad get <path-to-exercise-files>

You now have exactly the files the notebooks expect — and a record of where they came from.

---

## Pointers and resources

| Resource | Link |
|----------|------|
| **DataLad Handbook** | [handbook.datalad.org](http://handbook.datalad.org) |
| **OpenNeuro** | [openneuro.org](https://openneuro.org) |
| **DataLad datasets** | [datasets.datalad.org](http://datasets.datalad.org) |
| **MR-RATE dataset** | [huggingface.co/datasets/Forithmus/MR-RATE](https://huggingface.co/datasets/Forithmus/MR-RATE) |
| **MR-RATE code** | [github.com/forithmus/MR-RATE](https://github.com/forithmus/MR-RATE) |
| **This notebook** | [github.com/arush-arun/AS4SAN-Workshop](https://github.com/arush-arun/AS4SAN-Workshop) |

The DataLad Handbook is the single best starting point — it's a full tutorial with worked examples, no programming background required.